<a href="https://colab.research.google.com/github/m-ary-t/CA05---kNN-Recommender/blob/main/CA05_kNN_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library Downloads, Data Imports, & Data Cleaning

In [ ]:
# Core Imports
import pandas as pd
import numpy as np

# For Modeling
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


In [ ]:
df = pd.read_csv('https://github.com/ArinB/MSBA-CA-Data/raw/main/CA05/movies_recommendation_data.csv')
df

,Movie ID,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
0,58,The Imitation Game,8.0,1,1,1,0,0,0,0,0
1,8,Ex Machina,7.7,0,1,0,0,0,1,0,0
2,46,A Beautiful Mind,8.2,1,1,0,0,0,0,0,0
3,62,Good Will Hunting,8.3,0,1,0,0,0,0,0,0
4,97,Forrest Gump,8.8,0,1,0,0,0,0,0,0
5,98,21,6.8,0,1,0,0,1,0,1,0
6,31,Gifted,7.6,0,1,0,0,0,0,0,0
7,3,Travelling Salesman,5.9,0,1,0,0,0,1,0,0
8,51,Avatar,7.9,0,0,0,0,0,0,0,0
9,47,The Karate Kid,7.2,0,1,0,0,0,0,0,0


In [ ]:
df.columns

Index(['Movie ID', 'Movie Name', 'IMDB Rating', 'Biography', 'Drama',
       'Thriller', 'Comedy', 'Crime', 'Mystery', 'History', 'Label'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie ID     30 non-null     int64  
 1   Movie Name   30 non-null     object 
 2   IMDB Rating  30 non-null     float64
 3   Biography    30 non-null     int64  
 4   Drama        30 non-null     int64  
 5   Thriller     30 non-null     int64  
 6   Comedy       30 non-null     int64  
 7   Crime        30 non-null     int64  
 8   Mystery      30 non-null     int64  
 9   History      30 non-null     int64  
 10  Label        30 non-null     int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 2.7+ KB


# Building a kNN Recommender

## Inspecting and Selecting Features

We’ll create a copy of the orignal dataset without the labels column  (since the column is for classifier/regression and all their values are 0) and keep only numeric columns to compute similarity (so we will also exclude 'Movie Name' and 'Movie ID')

In [ ]:
# Keep only the numeric feature columns used for proximity in kNN
feature_cols = ['IMDB Rating', 'Biography', 'Drama',
                'Thriller', 'Comedy', 'Crime', 'Mystery', 'History']
X = df[feature_cols].copy()

X

,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History
0,8.0,1,1,1,0,0,0,0
1,7.7,0,1,0,0,0,1,0
2,8.2,1,1,0,0,0,0,0
3,8.3,0,1,0,0,0,0,0
4,8.8,0,1,0,0,0,0,0
5,6.8,0,1,0,0,1,0,1
6,7.6,0,1,0,0,0,0,0
7,5.9,0,1,0,0,0,1,0
8,7.9,0,0,0,0,0,0,0
9,7.2,0,1,0,0,0,0,0


In [ ]:
# Keep titles for reference
titles = df['Movie Name'].values if 'Movie Name' in df.columns else np.arange(len(df))

titles

array(['The Imitation Game', 'Ex Machina', 'A Beautiful Mind',
       'Good Will Hunting', 'Forrest Gump', '21', 'Gifted',
       'Travelling Salesman', 'Avatar', 'The Karate Kid',
       'A Brilliant Young Mind', 'A Time To Kill', 'Interstellar',
       'The Wolf of Wall Street', 'Black Panther', 'Inception',
       'The Wind Rises', 'Spirited Away', 'Finding Forrester',
       'The Fountain', 'The DaVinci Code', 'Stand and Deliver',
       'The Terminator', '21 Jump Street', 'The Avengers',
       'Thor: Ragnarok', 'Spirit: Stallion of the Cimarron',
       'Hacksaw Ridge', '12 Years a Slave', 'Queen of Katwe'],
      dtype=object)

## Standardized Scaling of Features

Since kNN works based on distance between data points, we need to standardize the data before training the model, which will help avoid problems due to scaling.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# "The Post" Example

## Fitting a Nearest Neighbor Model
We will use the **Brute Force Algorithm** since we have a small sample size of 30 movies and a small dimensionality of 9 variables for our kNN. We will use **Euclidean Distance** to calculate the distance between the to-be predicted point and its n_nearest neighbors.    

In [ ]:
# n_neighbors = 6 so we can drop the item itself later and still return top 5.
nn = NearestNeighbors(n_neighbors=6, metric='euclidean', algorithm='brute')
nn.fit(X_scaled)


NearestNeighbors(algorithm='brute', metric='euclidean', n_neighbors=6)

## Testing kNN Brute Force Algorithm

*"The Post"* :


*   **IMDB Rating** = 7.2
*   **Biography** = Yes
*   **Drama** = Yes
*   **Thriller** = No
*   **Comedy** = No
*   **Crime** = No
*   **Mystery** = No
*   **History** = Yes

In [ ]:
X.columns

Index(['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime',
       'Mystery', 'History'],
      dtype='object')

In [ ]:
# the order of the values for 'The Post' match the order of the columns
the_post_raw = np.array([[7.2, 1, 1, 0, 0, 0, 0, 1]])

# IMPORTANT: scale the query with the SAME scaler fit on X
the_post = scaler.transform(the_post_raw)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


### Retrieving the 5 most similar movies to "The Post"

In [ ]:
distances, indices = nn.kneighbors(the_post, n_neighbors=6)  # 6 to allow dropping self
indices = indices.flatten()
distances = distances.flatten()

# If your dataset already contains "The Post", the closest neighbor could be itself (distance ~ 0).
# We'll drop it if the title matches; otherwise, we just take the first 5.
recs = []
for idx, d in zip(indices, distances):
    title = titles[idx]
    # If the dataset includes "The Post", skip exact self-match:
    if isinstance(title, str) and title.strip().lower() == "the post":
        continue
    recs.append((title, float(d)))

# Keep the top 5
recs = recs[:5]

print("Top 5 'More Like This' for 'The Post':")
for rank, (title, d) in enumerate(recs, start=1):
    print(f"{rank}. {title} (distance: {d:.3f})")

Top 5 'More Like This' for 'The Post':
1. 12 Years a Slave (distance: 1.374)
2. Hacksaw Ridge (distance: 1.527)
3. Queen of Katwe (distance: 3.347)
4. The Wind Rises (distance: 3.457)
5. A Beautiful Mind (distance: 3.666)


### The Full Rows of the Top 5 Movies Most Similar to "The Post"

In [ ]:
neighbor_indices = [np.where(titles == r[0])[0][0] if isinstance(r[0], str) else r[0] for r in recs]
df.iloc[neighbor_indices][['Movie Name'] + feature_cols]

,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History
28,12 Years a Slave,8.1,1,1,0,0,0,0,1
27,Hacksaw Ridge,8.2,1,1,0,0,0,0,1
29,Queen of Katwe,7.4,1,1,0,0,0,0,0
16,The Wind Rises,7.8,1,1,0,0,0,0,0
2,A Beautiful Mind,8.2,1,1,0,0,0,0,0


# Conclusion:

We chose the **Brute Force Algorithm** for our kNN Model due to having a small sample size and small dimensionality, and we measured the distance between our data point for prediction with its nearest neighbors using **Euclidean Distance**.

The Top 5 Movies Most Similar to *"The Post"* using **Brute Force Algorithm** are:



1. 12 Years a Slave (distance: 1.374)
2. Hacksaw Ridge (distance: 1.527)
3. Queen of Katwe (distance: 3.347)
4. The Wind Rises (distance: 3.457)
5. A Beautiful Mind (distance: 3.666)



---


The recommended movies share strong similarity with “The Post,” particularly in the Biography and Drama genres, along with similar IMDB ratings. This indicates that the kNN model is effectively capturing both thematic and quality-based similarities.

---

A limitation of this model is that it only considers genres and ratings, and does not account for factors such as actors, directors, or storyline themes.